# Tarea en Intro00

## Presentado por: Eduardo Pérez

### Ejercicio 1: Carga de datos y preprocesamiento

Cargando y visualizando datos:

In [2]:
import pandas as pd
from datetime import date
import numpy as np

df_raw = pd.read_csv('kumpula-weather-2017.csv')

df_raw.head(10)

,Year,m,d,Time,Time zone,Precipitation amount (mm),Snow depth (cm),Air temperature (degC)
0,2017,1,1,00:00,UTC,-1.0,-1.0,0.6
1,2017,1,2,00:00,UTC,4.4,-1.0,-3.9
2,2017,1,3,00:00,UTC,6.6,7.0,-6.5
3,2017,1,4,00:00,UTC,-1.0,13.0,-12.8
4,2017,1,5,00:00,UTC,-1.0,10.0,-17.8
5,2017,1,6,00:00,UTC,0.3,10.0,-17.8
6,2017,1,7,00:00,UTC,5.3,10.0,-3.8
7,2017,1,8,00:00,UTC,-1.0,12.0,-0.5
8,2017,1,9,00:00,UTC,1.1,12.0,0.5
9,2017,1,10,00:00,UTC,0.3,9.0,1.7


Creación de la función "monthday_to_days" y de un data frame con copia de los datos para verificar que la función está trabajando correctamente.

In [3]:
df_working = df_raw.copy()
#print(df_working)

def monthday_to_day(month,day):
    start_date = date(2017,1,1)
    current_date = date(2017,int(month),int(day))
    return (current_date - start_date).days + 1

df_working['days'] = df_working.apply(lambda row: monthday_to_day (row['m'],row['d']), axis=1)

df_working.head()
    

,Year,m,d,Time,Time zone,Precipitation amount (mm),Snow depth (cm),Air temperature (degC),days
0,2017,1,1,00:00,UTC,-1.0,-1.0,0.6,1
1,2017,1,2,00:00,UTC,4.4,-1.0,-3.9,2
2,2017,1,3,00:00,UTC,6.6,7.0,-6.5,3
3,2017,1,4,00:00,UTC,-1.0,13.0,-12.8,4
4,2017,1,5,00:00,UTC,-1.0,10.0,-17.8,5


Encontrando y filtrando los valores NaN, inf o masks:

In [5]:
nan_counts = df_working.isna().sum()
inf_counts = df_working.isin([np.inf, -np.inf]).sum()

print("NaN Values information:")
print(nan_counts)
print("Inf Values information:")
print(inf_counts)

df_filtered = df_working.replace([np.inf, -np.inf], np.nan).dropna()
df_filtered = df_filtered.reset_index(drop=True)

print("Size of dataframe before and after filtering:")
print(df_working.shape[0])
print(df_filtered.shape[0])

NaN Values information:
Year                         0
m                            0
d                            0
Time                         0
Time zone                    0
Precipitation amount (mm)    0
Snow depth (cm)              7
Air temperature (degC)       0
days                         0
dtype: int64
Inf Values information:
Year                         0
m                            0
d                            0
Time                         0
Time zone                    0
Precipitation amount (mm)    0
Snow depth (cm)              0
Air temperature (degC)       0
days                         0
dtype: int64
Size of dataframe before and after filtering:
365
358


Calculando anomalías (temperatura - media móvil de 30 días):

In [6]:
temp_col_label = 'Air temperature (degC)'
df_filtered['Moving_Average_30'] = df_filtered[temp_col_label].rolling(window=30).mean()
df_filtered['Anomaly'] = df_filtered[temp_col_label]-df_filtered['Moving_Average_30']
df_filtered = df_filtered.dropna().reset_index(drop=True)

print(df_filtered.head())

   Year  m   d   Time Time zone  Precipitation amount (mm)  Snow depth (cm)  \
0  2017  1  30  00:00       UTC                        5.6              5.0   
1  2017  1  31  00:00       UTC                       -1.0              4.0   
2  2017  2   1  00:00       UTC                        1.5              4.0   
3  2017  2   2  00:00       UTC                        0.2              5.0   
4  2017  2   3  00:00       UTC                       -1.0              6.0   

   Air temperature (degC)  days  Moving_Average_30   Anomaly  
0                     1.0    30          -2.400000  3.400000  
1                     0.2    31          -2.413333  2.613333  
2                    -0.6    32          -2.303333  1.703333  
3                    -0.8    33          -2.113333  1.313333  
4                    -0.2    34          -1.693333  1.493333  


Detectando valores atípicos (método del IQR: Q1 - 1.5 * IQR). Resultado esperado: temps_clean (N x 2), days (N,):

In [11]:
Q1 = df_filtered['Anomaly'].quantile(0.25)
Q3 = df_filtered['Anomaly'].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR

df_clean = df_filtered[(df_filtered['Anomaly'] >= lower_bound) & (df_filtered['Anomaly'] <= upper_bound)]
temps_clean = df_clean[[temp_col_label,'Anomaly']].to_numpy()
days = df_clean['days'].to_numpy()

print(temps_clean[:5])
print(days[:5])

[[ 1.          3.4       ]
 [ 0.2         2.61333333]
 [-0.6         1.70333333]
 [-0.8         1.31333333]
 [-0.2         1.49333333]]
[30 31 32 33 34]
